In [ ]:
'''
windows antiberty_py311环境

1. 去掉没有heavy/light的细胞
2. 计算antiberty embedding

最终存储：ref3_bcr_antiberty_full-length_v2.npy
'''

In [ ]:
# Setup and Imports
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
adata = sc.read_h5ad('./data/Bcells_merged_BCR_palantir_pseudotime_v2.h5ad')

In [ ]:
# 创建筛选条件：Heavy和Light长度均大于80且无空值
mask = (
    adata.obs['Heavy'].notna() &           # Heavy 非空
    adata.obs['Light'].notna() &           # Light 非空
    (adata.obs['Heavy'].astype(str).str.len() >= 80) &   # Heavy长度>=80
    (adata.obs['Light'].astype(str).str.len() >= 80)     # Light长度>=80
)

print(f"原始细胞数: {adata.n_obs}")
print(f"满足条件的细胞数: {mask.sum()}")
print(f"被过滤掉的细胞数: {(~mask).sum()}")

# 筛选数据
adata = adata[mask].copy()

In [ ]:
# 确认'Heavy', 'Light'长度均大于80且无空值
ok = (adata.obs[['Heavy', 'Light']]          # 取两列
      .dropna()                       # 去掉任何 NaN
      .astype(str)                    # 确保字符串类型
      .applymap(len)                  # 计算每条序列长度
      .ge(80)                         # 大于 80 返回 True/False
      .all(axis=None))                # 全部 True 才返回 True

print(ok)   # True → 全部满足；False → 至少一条不满足

In [ ]:
from btraj.embedding.embed import AntiBERTyEmbedder
save_name = './data/ref3_'

embedder = AntiBERTyEmbedder(
    max_length=256,
    batch_size=256,
)
X_bcr_bert = embedder.compute_bcr_embeddings(
    adata,
    heavy_col="Heavy",
    light_col="Light",
    # pca_dim=256,                 # 或 384
    pca_dim=None,
    precomputed_path=save_name+ "bcr_antiberty_full-length_v2.npy",
)
